# Visualización interactiva — NEUROCON & Tao Wu

Explora volúmenes NIfTI preprocesados para identificar **slices relevantes**.

| Dataset | Ruta |
|---------|------|
| NEUROCON | `/workspace/IL-CLDM/data/neurocon/preprocessed` |
| Tao Wu | `/workspace/IL-CLDM/data/taowu/preprocessed` |

Solo se visualizan los volúmenes `03_resampled_final.nii.gz`.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
from ipywidgets import Dropdown, IntSlider, interact, interactive_output, VBox, HBox
from IPython.display import display

%matplotlib inline

DATASETS = {
    "neurocon": Path("/workspace/IL-CLDM/data/neurocon/preprocessed_standard"),
    "taowu": Path("/workspace/IL-CLDM/data/taowu/preprocessed_standard"),
}

NII_NAME = "01_brain_extracted.nii.gz"

for name, root in DATASETS.items():
    assert root.is_dir(), f"No existe: {root}"
    print(f"{name}: {root}")
print(f"Archivo visualizado: {NII_NAME}")

neurocon: /workspace/IL-CLDM/data/neurocon/preprocessed_standard
taowu: /workspace/IL-CLDM/data/taowu/preprocessed_standard
Archivo visualizado: 01_brain_extracted.nii.gz


## Inventario de casos

In [2]:
def list_subjects(dataset: str, group: str | None = None) -> list[str]:
    root = DATASETS[dataset]
    groups = [group] if group else ["control", "parkinson"]
    subjects = []
    for g in groups:
        gdir = root / g
        if not gdir.is_dir():
            continue
        for p in sorted(gdir.iterdir()):
            if p.is_dir() and p.name.startswith("sub-") and (p / NII_NAME).is_file():
                subjects.append(p.name)
    return subjects


def subject_group(dataset: str, subject_id: str) -> str:
    root = DATASETS[dataset]
    for g in ("control", "parkinson"):
        if (root / g / subject_id).is_dir():
            return g
    raise FileNotFoundError(f"{subject_id} no encontrado en {dataset}")


def resolve_nii_path(dataset: str, subject_id: str) -> Path:
    g = subject_group(dataset, subject_id)
    path = DATASETS[dataset] / g / subject_id / NII_NAME
    if not path.is_file():
        raise FileNotFoundError(path)
    return path


def load_volume(path: Path) -> np.ndarray:
    vol = nib.load(str(path)).get_fdata(dtype=np.float32)
    # nibabel puede devolver (X,Y,Z) o más dims; tomar 3D
    while vol.ndim > 3:
        vol = vol[..., 0]
    return vol


rows = []
for ds, root in DATASETS.items():
    for g in ("control", "parkinson"):
        gdir = root / g
        if not gdir.is_dir():
            continue
        n = sum(
            1
            for p in gdir.iterdir()
            if p.is_dir() and p.name.startswith("sub-") and (p / NII_NAME).is_file()
        )
        rows.append({"dataset": ds, "group": g, "n_subjects": n})

summary = pd.DataFrame(rows)
display(summary)
print(f"Total sujetos: {summary['n_subjects'].sum()}")

,dataset,group,n_subjects
0,neurocon,control,16
1,neurocon,parkinson,27
2,taowu,control,20
3,taowu,parkinson,20


Total sujetos: 83


## Visor interactivo (corte axial)

Usa los dropdowns para elegir dataset / grupo / sujeto (`03_resampled_final`), y el slider `z` para recorrer los cortes.
El índice actual se imprime debajo del plot para anotar slices relevantes.

In [ ]:
# Estado del volumen cargado (evita releer el NIfTI en cada movimiento del slider)
_state = {"path": None, "vol": None, "info": ""}


def _ensure_volume(dataset: str, group: str, subject: str) -> np.ndarray:
    path = DATASETS[dataset] / group / subject / NII_NAME
    if not path.is_file():
        raise FileNotFoundError(path)
    if _state["path"] != path:
        vol = load_volume(path)
        _state["path"] = path
        _state["vol"] = vol
        _state["info"] = (
            f"{dataset} | {group} | {subject} | {NII_NAME}\n"
            f"shape={tuple(vol.shape)}  "
            f"min={vol.min():.3g}  max={vol.max():.3g}  mean={vol.mean():.3g}"
        )
        print(f"Cargado: {path}")
        print(_state["info"])
    return _state["vol"]


def ver_corte(dataset, group, subject, z=0):
    vol = _ensure_volume(dataset, group, subject)
    z = int(np.clip(z, 0, vol.shape[2] - 1))
    slice_2d = vol[:, :, z].T  # orientación neurológica habitual en axial

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(slice_2d, cmap="gray", origin="lower")
    ax.set_title(f"z = {z} / {vol.shape[2] - 1}")
    ax.axis("off")
    plt.tight_layout()
    plt.show()
    print(f"Slice relevante candidato → z={z}  |  {_state['path'].name}")


# Widgets enlazados
w_dataset = Dropdown(options=list(DATASETS.keys()), value="neurocon", description="Dataset:")
w_group = Dropdown(options=["control", "parkinson"], value="control", description="Grupo:")
w_subject = Dropdown(options=list_subjects("neurocon", "control"), description="Sujeto:")
w_z = IntSlider(value=64, min=0, max=127, step=1, description="z:", continuous_update=False)


def _refresh_subjects(*_):
    subs = list_subjects(w_dataset.value, w_group.value)
    w_subject.options = subs
    if subs and w_subject.value not in subs:
        w_subject.value = subs[0]


def _refresh_z_range(*_):
    try:
        vol = _ensure_volume(w_dataset.value, w_group.value, w_subject.value)
        w_z.max = max(0, vol.shape[2] - 1)
        w_z.value = min(w_z.value, w_z.max)
    except Exception as e:
        print(e)


w_dataset.observe(_refresh_subjects, names="value")
w_group.observe(_refresh_subjects, names="value")
w_subject.observe(_refresh_z_range, names="value")

_refresh_subjects()
_refresh_z_range()

ui = VBox([HBox([w_dataset, w_group]), w_subject, w_z])
out = interactive_output(
    ver_corte,
    {
        "dataset": w_dataset,
        "group": w_group,
        "subject": w_subject,
        "z": w_z,
    },
)
display(ui, out)

## Visor simple (estilo mínimo)

Versión cercana al snippet original: carga un volumen fijo y solo mueve el slider `z`.

In [ ]:
# Cambia estos valores y re-ejecuta la celda
DATASET = "taowu"          # "neurocon" | "taowu"
GROUP = "parkinson"        # "control" | "parkinson"
SUBJECT = None             # None = primer sujeto del grupo

subs = list_subjects(DATASET, GROUP)
SUBJECT = SUBJECT or subs[0]
nii_path = DATASETS[DATASET] / GROUP / SUBJECT / NII_NAME
volumen = load_volume(nii_path)

print(f"Archivo: {nii_path}")
print(f"shape={volumen.shape}  dtype={volumen.dtype}")


def ver_corte_simple(z=0):
    plt.figure(figsize=(5, 5))
    plt.imshow(volumen[:, :, z].T, cmap="gray", origin="lower")
    plt.title(f"{SUBJECT}  |  z={z}")
    plt.axis("off")
    plt.show()


interact(ver_corte_simple, z=(0, volumen.shape[2] - 1))

## Tres planos (axial / coronal / sagital)

Útil para confirmar que el slice axial elegido atraviesa la anatomía de interés (p. ej. ganglios basales).

In [ ]:
def ver_tres_planos(dataset, group, subject, z=0, y=0, x=0):
    vol = _ensure_volume(dataset, group, subject)
    z = int(np.clip(z, 0, vol.shape[2] - 1))
    y = int(np.clip(y, 0, vol.shape[1] - 1))
    x = int(np.clip(x, 0, vol.shape[0] - 1))

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(vol[:, :, z].T, cmap="gray", origin="lower")
    axes[0].set_title(f"Axial z={z}")
    axes[1].imshow(vol[:, y, :].T, cmap="gray", origin="lower")
    axes[1].set_title(f"Coronal y={y}")
    axes[2].imshow(vol[x, :, :].T, cmap="gray", origin="lower")
    axes[2].set_title(f"Sagital x={x}")
    for ax in axes:
        ax.axis("off")
    fig.suptitle(f"{dataset} | {group} | {subject} | {NII_NAME}", fontsize=11)
    plt.tight_layout()
    plt.show()
    print(f"Índices → axial z={z}, coronal y={y}, sagital x={x}")


w_z3 = IntSlider(value=64, min=0, max=127, step=1, description="z (axial):", continuous_update=False)
w_y3 = IntSlider(value=64, min=0, max=127, step=1, description="y (cor):", continuous_update=False)
w_x3 = IntSlider(value=64, min=0, max=127, step=1, description="x (sag):", continuous_update=False)


def _sync_xyz_ranges(*_):
    try:
        vol = _ensure_volume(w_dataset.value, w_group.value, w_subject.value)
        w_z3.max = max(0, vol.shape[2] - 1)
        w_y3.max = max(0, vol.shape[1] - 1)
        w_x3.max = max(0, vol.shape[0] - 1)
        w_z3.value = min(w_z3.value, w_z3.max)
        w_y3.value = min(w_y3.value, w_y3.max)
        w_x3.value = min(w_x3.value, w_x3.max)
    except Exception as e:
        print(e)


for w in (w_dataset, w_group, w_subject):
    w.observe(_sync_xyz_ranges, names="value")

_sync_xyz_ranges()

ui3 = VBox([HBox([w_dataset, w_group]), w_subject, w_z3, w_y3, w_x3])
out3 = interactive_output(
    ver_tres_planos,
    {
        "dataset": w_dataset,
        "group": w_group,
        "subject": w_subject,
        "z": w_z3,
        "y": w_y3,
        "x": w_x3,
    },
)
display(ui3, out3)

## Anotar slices relevantes

Cuando encuentres un corte útil, añádelo a la lista y exporta el CSV al final.

In [ ]:
from datetime import datetime

relevant_slices: list[dict] = []


def anotar_slice_actual(nota: str = ""):
    """Registra el z actual del visor principal (w_z) junto con el caso cargado."""
    if _state["vol"] is None or _state["path"] is None:
        print("Primero carga un volumen en el visor interactivo.")
        return
    entry = {
        "dataset": w_dataset.value,
        "group": w_group.value,
        "subject": w_subject.value,
        "file": NII_NAME,
        "path": str(_state["path"]),
        "z": int(w_z.value),
        "shape": str(tuple(_state["vol"].shape)),
        "nota": nota,
        "timestamp": datetime.now().isoformat(timespec="seconds"),
    }
    relevant_slices.append(entry)
    print(f"Añadido: {entry['subject']} z={entry['z']}  ({entry['nota'] or 'sin nota'})")


# Ejemplo: anotar el slice actual del slider del visor principal
anotar_slice_actual(nota="candidato")
display(pd.DataFrame(relevant_slices))

In [ ]:
# Exportar anotaciones (opcional)
out_csv = Path("/workspace/IL-CLDM/code/neurocon_tao_relevant_slices.csv")
if relevant_slices:
    pd.DataFrame(relevant_slices).to_csv(out_csv, index=False)
    print(f"Guardado: {out_csv}")
else:
    print("No hay anotaciones aún. Usa anotar_slice_actual(...).")

## Visor de un NIfTI cualquiera

Mismo visor de tres planos (axial / coronal / sagital). Pega la ruta de cualquier `.nii` o `.nii.gz` y pulsa **Cargar** (o Enter en el campo de texto).

In [ ]:
from ipywidgets import Button, Text, Layout

_free_state = {"path": None, "vol": None}

w_nii_path = Text(
    value="/workspace/IL-CLDM/tmp_mprage/sub-3010.nii.gz",
    placeholder="",
    description="NIfTI:",
    continuous_update=False,
    layout=Layout(width="80%"),
    style={"description_width": "initial"},
)
w_load_nii = Button(description="Cargar", button_style="primary")
w_z_free = IntSlider(value=0, min=0, max=0, step=1, description="z (axial):", continuous_update=False)
w_y_free = IntSlider(value=0, min=0, max=0, step=1, description="y (cor):", continuous_update=False)
w_x_free = IntSlider(value=0, min=0, max=0, step=1, description="x (sag):", continuous_update=False)


def _set_slider(slider, n: int, center: bool) -> None:
    hi = max(0, n - 1)
    slider.value = min(slider.value, hi)
    slider.max = hi
    if center:
        slider.value = hi // 2


def _load_free_nii(path_str: str) -> np.ndarray | None:
    raw = (path_str or "").strip()
    if not raw:
        print("Escribe la ruta de un archivo .nii o .nii.gz y pulsa Cargar.")
        return None
    path = Path(raw).expanduser()
    name = path.name.lower()
    if not (name.endswith(".nii") or name.endswith(".nii.gz")):
        print(f"No parece un NIfTI: {path}")
        return None
    if not path.is_file():
        print(f"No existe: {path}")
        return None
    if _free_state["path"] != path:
        vol = load_volume(path)
        if vol.ndim != 3:
            print(f"Se esperaba un volumen 3D, shape={tuple(vol.shape)}")
            return None
        _free_state["path"] = path
        _free_state["vol"] = vol
        print(f"Cargado: {path}")
        print(
            f"shape={tuple(vol.shape)}  "
            f"min={vol.min():.3g}  max={vol.max():.3g}  mean={vol.mean():.3g}"
        )
        _set_slider(w_z_free, vol.shape[2], center=True)
        _set_slider(w_y_free, vol.shape[1], center=True)
        _set_slider(w_x_free, vol.shape[0], center=True)
    return _free_state["vol"]


def ver_nii_libre(z=0, y=0, x=0):
    vol = _free_state["vol"]
    if vol is None:
        print("Ningún volumen cargado.")
        return
    z = int(np.clip(z, 0, vol.shape[2] - 1))
    y = int(np.clip(y, 0, vol.shape[1] - 1))
    x = int(np.clip(x, 0, vol.shape[0] - 1))

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(vol[:, :, z].T, cmap="gray", origin="lower")
    axes[0].set_title(f"Axial z={z}")
    axes[1].imshow(vol[:, y, :].T, cmap="gray", origin="lower")
    axes[1].set_title(f"Coronal y={y}")
    axes[2].imshow(vol[x, :, :].T, cmap="gray", origin="lower")
    axes[2].set_title(f"Sagital x={x}")
    for ax in axes:
        ax.axis("off")
    fig.suptitle(str(_free_state["path"]), fontsize=10)
    plt.tight_layout()
    plt.show()
    print(f"Índices → axial z={z}, coronal y={y}, sagital x={x}")


def _on_load_nii(_=None):
    _load_free_nii(w_nii_path.value)


w_load_nii.on_click(_on_load_nii)
w_nii_path.observe(lambda _: _on_load_nii(), names="value")

ui_free = VBox([HBox([w_nii_path, w_load_nii]), w_z_free, w_y_free, w_x_free])
out_free = interactive_output(
    ver_nii_libre,
    {"z": w_z_free, "y": w_y_free, "x": w_x_free},
)
display(ui_free, out_free)

Output()